# KOWAS-EPI fitting 기간 EDA (2단계)

**목적**: `folds.csv`의 10개 평가 창 각각의 `train_until` 시점을 fitting 기간 상한으로 삼아, 그 시점까지 공개된 정보만으로
(1) 지역별 기술통계, (2) README §5-4 검정 A(강수 희석 부분상관), (3) 검정 B(기상 추가 설명력 ΔR²),
(4) README §6 하수-임상 교차상관, (5) `plant_meta.csv` 지역 특성 탐색을 fold별로 반복 재현한다.

**가정 및 전제**
- fitting 기간 = 각 fold의 `train_until` 이전(포함) 데이터. 평가 창(target_start~target_end) 기간의 라벨/원자료는 어떤 fold에서도 들여다보지 않는다.
- `plant_meta.csv`는 `first_sample_week ≤ train_until`인 처리장만 사용하고, `in_current_registry`·`last_sample_week`는 미래 시점 정보라 전혀 사용하지 않는다.
- 로그 변환 시 0/결측은 사전에 걸러낸 표본만 사용한다(README와 동일 원칙).
- 이 노트북은 채점용이 아니라 탐색적 분석이므로, fold별로 표본 크기가 다르고 초기 fold는 표본이 매우 작다는 점을 결과 해석에 반영한다.


In [1]:
import sys, math, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

import matplotlib
matplotlib.use("Agg")
import matplotlib.font_manager as fm
_font_path = str(Path.home() / ".local/share/fonts/NanumGothic-Regular.ttf")
fm.fontManager.addfont(_font_path)
matplotlib.rcParams["font.family"] = [fm.FontProperties(fname=_font_path).get_name(), "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

ROOT = Path("/home/geon1/github/kowas-epi-forecast")
KOWAS = ROOT / "KOWAS-EPI"
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## 0. 데이터 로드 & 주차 유틸리티

`evaluation.py`와 동일한 `(year, week)` 튜플 비교로 주차 순서를 판단한다(문자열 정렬 금지 — 예: "2023-W9" 같은 두 자리 미만 표기가 없어 실제로는 안전하지만, 명시적으로 연-주 비교를 쓴다).

In [2]:
def parse(k):
    y, w = k.split("-W")
    return int(y), int(w)

def le(a, b):
    """date_week a <= b"""
    return parse(a) <= parse(b)

def shift_week(w, k):
    y, wk = parse(w)
    d = datetime.date.fromisocalendar(y, wk, 1) + datetime.timedelta(weeks=k)
    iso = d.isocalendar()
    return f"{iso[0]}-W{iso[1]:02d}"

def filter_until(df, t_max, col="date_week"):
    mask = df[col].apply(lambda w: le(w, t_max))
    return df.loc[mask].copy()

panel = pd.read_csv(KOWAS / "kowas_epi_panel.csv", dtype=str, keep_default_na=False)
national = pd.read_csv(KOWAS / "national_weekly.csv", dtype=str, keep_default_na=False)
plant = pd.read_csv(KOWAS / "plant_meta.csv", dtype=str, keep_default_na=False)
folds = pd.read_csv(KOWAS / "folds.csv", dtype=str, keep_default_na=False)

num_cols = ["n_sites","conc_mean","conc_log10","conc_3wk_avg","conc_base_avg",
            "wow_change_rate","precip_mm","temp_avg","pop_served","pop_sampled"]
panel_num = panel.copy()
for c in num_cols:
    panel_num[c] = pd.to_numeric(panel_num[c].replace("", np.nan))
panel_num["week_no"] = panel_num["date_week"].str.split("-W").str[1].astype(int)

nat_num = national.copy()
for c in ["conc_mean_national","conc_3wk_avg","wow_change_rate","n_samples","covid_cases_national"]:
    nat_num[c] = pd.to_numeric(nat_num[c].replace("", np.nan))
nat_num["week_no"] = nat_num["date_week"].str.split("-W").str[1].astype(int)

plant_num = plant.copy()
plant_num["treatment_population"] = pd.to_numeric(plant_num["treatment_population"].replace("", np.nan))

folds[["fold_id", "train_until"]]

,fold_id,train_until
0,2024-Q2,2024-W12
1,2024-Q3,2024-W25
2,2024-Q4,2024-W38
3,2025-Q1,2024-W51
4,2025-Q2,2025-W12
5,2025-Q3,2025-W25
6,2025-Q4,2025-W38
7,2026-Q1,2025-W51
8,2026-Q2,2026-W12
9,2026-Q3,2026-W25


## 1. 기술통계 — 지역별 `conc_mean` (마지막 fold, 가장 많은 데이터) & 전국 추이

In [3]:
last_fold = folds.iloc[-1]
t_last = last_fold["train_until"]
p_last = filter_until(panel_num, t_last)

desc = p_last.groupby("region").agg(
    conc_mean_mean=("conc_mean", "mean"),
    conc_mean_std=("conc_mean", "std"),
    n_weeks=("conc_mean", "size"),
)
missing_by_region = p_last[p_last["conc_mean"].isna()].groupby("region").size().rename("n_missing")
desc = desc.join(missing_by_region).fillna({"n_missing": 0})
desc["n_missing"] = desc["n_missing"].astype(int)
desc = desc.sort_values("conc_mean_mean", ascending=False)
desc.to_csv(ROOT / "notebooks" / "_desc_region_last_fold.csv")
print(f"fold={last_fold['fold_id']}  train_until={t_last}  region 수={desc.shape[0]}")
desc.round(1)

fold=2026-Q3  train_until=2026-W25  region 수=17


,conc_mean_mean,conc_mean_std,n_weeks,n_missing
region,,,,
대구,1584203.0,5179764.2,168,11
세종,466092.2,1217373.8,168,9
부산,316662.5,866943.1,168,6
울산,233752.5,695721.4,168,18
경남,143609.6,1244575.5,168,23
대전,53991.1,104037.1,168,5
경기,42581.9,109004.2,168,6
충남,42176.2,90062.6,168,10
강원,35091.3,123421.0,168,7


In [4]:
n_last = filter_until(nat_num, t_last).sort_values("date_week", key=lambda s: s.map(parse))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(range(len(n_last)), n_last["conc_mean_national"], color="#1b6ca8", lw=1.5, label="conc_mean_national")
ax.plot(range(len(n_last)), n_last["conc_3wk_avg"], color="#e07b39", lw=1.2, ls="--", label="conc_3wk_avg")
xticks_idx = list(range(0, len(n_last), 13))
ax.set_xticks(xticks_idx)
ax.set_xticklabels([n_last["date_week"].iloc[i] for i in xticks_idx], rotation=45, ha="right", fontsize=8)
for _, frow in folds.iterrows():
    tmax = frow["train_until"]
    matches = n_last.index[n_last["date_week"] == tmax]
    if len(matches):
        pos = n_last.index.get_loc(matches[0])
        ax.axvline(pos, color="gray", alpha=0.3, lw=0.8)
ax.set_yscale("log")
ax.set_ylabel("전국 하수 농도 (copies/mL, log scale)")
ax.set_xlabel("주차 (date_week)")
ax.set_title(f"전국 평균 하수 농도 추이 (fitting 가능 구간, ~{t_last})\n회색 세로선 = 각 fold의 train_until")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG / "01_national_trend.png", dpi=150)
plt.show()

## 2. 검정 A·B — README §5-4 재현 (fold별 반복)

- **검정 A**: 종속 `log10(conc_mean/conc_3wk_avg)`, 설명 `log(1+precip_mm)`, 통제 `시·도더미 + sin/cos(2π·주차/52)` — 통제변수에 대한 잔차끼리 피어슨 상관(부분상관). README 전체데이터 값: r = −0.007
- **검정 B**: 기저 `conc_log10 ~ 시·도더미 + sin/cos(계절) + log10(1+conc_3wk_avg)`, 확장 모형에 `log(1+precip_mm)`+`temp_avg` 추가. ΔR² = 확장 R² − 기저 R². README 전체데이터 값: ΔR² = +0.0002

In [5]:
results_A, results_B = [], []

for _, frow in folds.iterrows():
    fold_id = frow["fold_id"]; t_max = frow["train_until"]
    p = filter_until(panel_num, t_max)

    # Test A
    sub = p[(p["conc_mean"] > 0) & (p["conc_3wk_avg"] > 0) & p["precip_mm"].notna()].copy()
    sub["y"] = np.log10(sub["conc_mean"] / sub["conc_3wk_avg"])
    sub["x"] = np.log(1 + sub["precip_mm"])
    sub["sin_w"] = np.sin(2*np.pi*sub["week_no"]/52)
    sub["cos_w"] = np.cos(2*np.pi*sub["week_no"]/52)
    n_a = len(sub); r_a = np.nan
    if n_a > 20 and sub["region"].nunique() > 1:
        my = smf.ols("y ~ C(region) + sin_w + cos_w", data=sub).fit()
        mx = smf.ols("x ~ C(region) + sin_w + cos_w", data=sub).fit()
        r_a = float(np.corrcoef(my.resid, mx.resid)[0, 1])
    results_A.append({"fold_id": fold_id, "train_until": t_max, "n": n_a, "r_partial": r_a})

    # Test B
    subB = p[(p["conc_log10"].notna()) & (p["conc_3wk_avg"] > 0) & p["precip_mm"].notna() & p["temp_avg"].notna()].copy()
    subB["sin_w"] = np.sin(2*np.pi*subB["week_no"]/52)
    subB["cos_w"] = np.cos(2*np.pi*subB["week_no"]/52)
    subB["log_c3"] = np.log10(1 + subB["conc_3wk_avg"])
    subB["log_precip"] = np.log(1 + subB["precip_mm"])
    n_b = len(subB); dr2 = np.nan
    if n_b > 20 and subB["region"].nunique() > 1:
        base = smf.ols("conc_log10 ~ C(region) + sin_w + cos_w + log_c3", data=subB).fit()
        ext = smf.ols("conc_log10 ~ C(region) + sin_w + cos_w + log_c3 + log_precip + temp_avg", data=subB).fit()
        dr2 = float(ext.rsquared - base.rsquared)
    results_B.append({"fold_id": fold_id, "train_until": t_max, "n": n_b, "delta_r2": dr2})

dfA = pd.DataFrame(results_A); dfB = pd.DataFrame(results_B)
dfA.to_csv(ROOT / "notebooks" / "_test_A_by_fold.csv", index=False)
dfB.to_csv(ROOT / "notebooks" / "_test_B_by_fold.csv", index=False)
print("검정 A (부분상관):"); display(dfA)
print("검정 B (ΔR²):"); display(dfB)

검정 A (부분상관):


,fold_id,train_until,n,r_partial
0,2024-Q2,2024-W12,763,0.025571
1,2024-Q3,2024-W25,984,0.026825
2,2024-Q4,2024-W38,1203,0.000441
3,2025-Q1,2024-W51,1423,-0.017204
4,2025-Q2,2025-W12,1610,-0.020589
5,2025-Q3,2025-W25,1831,-0.018963
6,2025-Q4,2025-W38,2052,-0.022914
7,2026-Q1,2025-W51,2272,-0.018753
8,2026-Q2,2026-W12,2459,-0.018174
9,2026-Q3,2026-W25,2667,-0.011687


검정 B (ΔR²):


,fold_id,train_until,n,delta_r2
0,2024-Q2,2024-W12,763,0.000272
1,2024-Q3,2024-W25,984,0.000546
2,2024-Q4,2024-W38,1203,0.000794
3,2025-Q1,2024-W51,1423,0.001159
4,2025-Q2,2025-W12,1610,0.000905
5,2025-Q3,2025-W25,1831,0.000644
6,2025-Q4,2025-W38,2052,0.000419
7,2026-Q1,2025-W51,2272,0.000308
8,2026-Q2,2026-W12,2459,0.000269
9,2026-Q3,2026-W25,2667,0.000247


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
order = folds["fold_id"].tolist()
dfA2 = dfA.set_index("fold_id").loc[order]
dfB2 = dfB.set_index("fold_id").loc[order]

axes[0].plot(range(len(order)), dfA2["r_partial"], marker="o", color="#1b6ca8")
axes[0].axhline(-0.007, color="#c0392b", ls="--", lw=1, label="README 전체데이터 r=-0.007")
axes[0].axhline(0, color="gray", lw=0.6)
axes[0].set_xticks(range(len(order))); axes[0].set_xticklabels(order, rotation=45, ha="right", fontsize=8)
axes[0].set_title("검정 A: 강수 희석 부분상관 (fold별)")
axes[0].set_ylabel("partial r")
axes[0].legend(frameon=False, fontsize=8)

axes[1].plot(range(len(order)), dfB2["delta_r2"], marker="o", color="#1b6ca8")
axes[1].axhline(0.0002, color="#c0392b", ls="--", lw=1, label="README 전체데이터 ΔR²=+0.0002")
axes[1].axhline(0, color="gray", lw=0.6)
axes[1].set_xticks(range(len(order))); axes[1].set_xticklabels(order, rotation=45, ha="right", fontsize=8)
axes[1].set_title("검정 B: 기상 추가 설명력 ΔR² (fold별)")
axes[1].set_ylabel("ΔR²")
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "02_testA_testB_by_fold.png", dpi=150)
plt.show()

## 3. 하수-임상 교차상관 — README §6 재현 (fold별 반복)

`national_weekly.csv`의 `conc_mean_national`·`covid_cases_national`을 log10 변환 후, 시차 k∈{-2,...,4}마다 "하수 t주 vs 임상 t+k주"를 같은 주차 키로 매칭해 피어슨 상관을 계산한다. `covid_cases_national`은 2024-W01부터 존재하므로 이른 fold(2024-Q2 등)는 표본이 매우 작다.

In [7]:
results_X = []
for _, frow in folds.iterrows():
    fold_id = frow["fold_id"]; t_max = frow["train_until"]
    n = filter_until(nat_num, t_max)
    nn = n[(n["conc_mean_national"].notna()) & (n["covid_cases_national"].notna())].copy()
    nn["log_conc"] = np.log10(nn["conc_mean_national"])
    nn["log_covid"] = np.log10(nn["covid_cases_national"])
    idx = nn.set_index("date_week")
    weeks_sorted = sorted(nn["date_week"].tolist(), key=parse)
    for k in [-2, -1, 0, 1, 2, 3, 4]:
        pairs = []
        for w in weeks_sorted:
            wk_k = shift_week(w, k)
            if wk_k in idx.index:
                pairs.append((idx.loc[w, "log_conc"], idx.loc[wk_k, "log_covid"]))
        n_k = len(pairs); r_k = np.nan
        if n_k > 5:
            a = np.array([x[0] for x in pairs]); b = np.array([x[1] for x in pairs])
            r_k = float(np.corrcoef(a, b)[0, 1])
        results_X.append({"fold_id": fold_id, "train_until": t_max, "lag": k, "n": n_k, "r": r_k})

dfX = pd.DataFrame(results_X)
dfX.to_csv(ROOT / "notebooks" / "_xcorr_by_fold.csv", index=False)
pivot_r = dfX.pivot(index="train_until", columns="lag", values="r").loc[folds["train_until"].tolist()]
pivot_n = dfX.pivot(index="train_until", columns="lag", values="n").loc[folds["train_until"].tolist()]
print("r:"); display(pivot_r.round(3))
print("n:"); display(pivot_n)

r:


lag,-2,-1,0,1,2,3,4
train_until,,,,,,,
2024-W12,0.154,0.194,0.333,0.523,0.347,0.460,0.249
2024-W25,0.906,0.935,0.963,0.966,0.948,0.922,0.878
2024-W38,0.783,0.907,0.960,0.931,0.838,0.707,0.548
2024-W51,0.838,0.927,0.966,0.950,0.881,0.771,0.632
2025-W12,0.820,0.919,0.882,0.877,0.840,0.754,0.654
2025-W25,0.811,0.910,0.882,0.873,0.839,0.752,0.651
2025-W38,0.814,0.914,0.886,0.874,0.831,0.737,0.626
2025-W51,0.786,0.886,0.875,0.869,0.832,0.743,0.636
2026-W12,0.823,0.905,0.896,0.890,0.858,0.788,0.700


n:


lag,-2,-1,0,1,2,3,4
train_until,,,,,,,
2024-W12,10,11,12,11,10,9,8
2024-W25,23,24,25,24,23,22,21
2024-W38,36,37,38,37,36,35,34
2024-W51,49,50,51,50,49,48,47
2025-W12,60,61,63,61,60,59,58
2025-W25,73,74,76,74,73,72,71
2025-W38,86,87,89,87,86,85,84
2025-W51,99,100,102,100,99,98,97
2026-W12,110,111,114,111,110,109,108


In [8]:
fig, ax = plt.subplots(figsize=(7.5, 5))
im = ax.imshow(pivot_r.values, aspect="auto", cmap="RdYlBu_r", vmin=0, vmax=1)
ax.set_xticks(range(len(pivot_r.columns))); ax.set_xticklabels(pivot_r.columns)
ax.set_yticks(range(len(pivot_r.index))); ax.set_yticklabels(pivot_r.index)
ax.set_xlabel("lag k (하수 t주 vs 임상 t+k주)")
ax.set_ylabel("fold train_until")
for i in range(pivot_r.shape[0]):
    for j in range(pivot_r.shape[1]):
        v = pivot_r.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                     color="white" if v > 0.6 else "black")
fig.colorbar(im, ax=ax, label="Pearson r (log10 conc vs log10 covid cases)")
ax.set_title("하수-임상 교차상관 r, fold(train_until)별 × lag별")
fig.tight_layout()
fig.savefig(FIG / "03_xcorr_heatmap.png", dpi=150)
plt.show()

## 4. `plant_meta.csv` 지역 특성 탐색

각 fold의 `train_until` 시점에 `first_sample_week ≤ train_until`인 처리장만으로 시·도별 처리장 개수·처리인구 합을 구하고, 같은 fold의 시·도별 평균/표준편차 `conc_mean`과의 탐색적 상관을 본다(과잉해석 금지 — 인과관계 아님).

In [9]:
plant_explore = []
for _, frow in folds.iterrows():
    fold_id = frow["fold_id"]; t_max = frow["train_until"]
    pm = plant_num[plant_num["first_sample_week"].apply(lambda w: le(w, t_max) if w else False)]
    plant_agg = pm.groupby("region").agg(n_plants=("plant_id", "size"),
                                          pop_sum=("treatment_population", "sum"))
    p = filter_until(panel_num, t_max)
    conc_agg = p.groupby("region")["conc_mean"].agg(conc_mean_avg="mean", conc_mean_std="std")
    merged = plant_agg.join(conc_agg, how="inner")
    merged["fold_id"] = fold_id
    plant_explore.append(merged.reset_index())

plant_explore_df = pd.concat(plant_explore, ignore_index=True)
plant_explore_df.to_csv(ROOT / "notebooks" / "_plant_explore_by_fold.csv", index=False)

last_pe = plant_explore_df[plant_explore_df["fold_id"] == folds.iloc[-1]["fold_id"]].dropna()
r_plants_mean = np.corrcoef(last_pe["n_plants"], last_pe["conc_mean_avg"])[0, 1]
r_pop_mean = np.corrcoef(last_pe["pop_sum"], last_pe["conc_mean_avg"])[0, 1]
r_plants_std = np.corrcoef(last_pe["n_plants"], last_pe["conc_mean_std"])[0, 1]
print(f"마지막 fold({folds.iloc[-1]['fold_id']}, train_until={t_last}) 기준:")
print(f"  corr(n_plants, conc_mean_avg) = {r_plants_mean:.3f}")
print(f"  corr(pop_sum, conc_mean_avg)  = {r_pop_mean:.3f}")
print(f"  corr(n_plants, conc_mean_std) = {r_plants_std:.3f}")
display(last_pe.round(1))

마지막 fold(2026-Q3, train_until=2026-W25) 기준:
  corr(n_plants, conc_mean_avg) = -0.381
  corr(pop_sum, conc_mean_avg)  = -0.088
  corr(n_plants, conc_mean_std) = -0.380


,region,n_plants,pop_sum,conc_mean_avg,conc_mean_std,fold_id
153,강원,6,816637,35091.3,123421.0,2026-Q3
154,경기,13,6608491,42581.9,109004.2,2026-Q3
155,경남,6,1723394,143609.6,1244575.5,2026-Q3
156,경북,10,1287477,25708.4,48929.4,2026-Q3
157,광주,3,691807,10324.6,32036.0,2026-Q3
158,대구,4,1948639,1584203.0,5179764.2,2026-Q3
159,대전,7,4353035,53991.1,104037.1,2026-Q3
160,부산,6,2961386,316662.5,866943.1,2026-Q3
161,서울,5,11268168,27129.3,50532.5,2026-Q3
162,세종,4,343577,466092.2,1217373.8,2026-Q3


In [10]:
fig, ax = plt.subplots(figsize=(6, 5))
sizes = (last_pe["pop_sum"] / last_pe["pop_sum"].max() * 400 + 30)
ax.scatter(last_pe["n_plants"], last_pe["conc_mean_avg"], s=sizes, c="#1b6ca8", alpha=0.7, edgecolor="white")
for _, row in last_pe.iterrows():
    ax.annotate(row["region"], (row["n_plants"], row["conc_mean_avg"]), fontsize=7, alpha=0.8,
                xytext=(3, 3), textcoords="offset points")
ax.set_yscale("log")
ax.set_xlabel(f"시·도 내 유효 처리장 수 (first_sample_week \u2264 {t_last})")
ax.set_ylabel("conc_mean 평균 (log scale)")
ax.set_title(f"처리장 수·처리인구(점 크기) vs 평균 농도 (fold={folds.iloc[-1]['fold_id']})")
fig.tight_layout()
fig.savefig(FIG / "04_plant_meta_explore.png", dpi=150)
plt.show()

## 5. 요약

- 검정 A·B, 교차상관 모두 fold(=시간)에 따라 값이 변하며, 가장 데이터가 많은 마지막 fold(2026-Q3, train_until=2026-W25)가 README 전체데이터 값에 가장 근접하는 경향을 보인다(정확히 같지는 않음 — README는 2026-W36까지, 이 fold는 2026-W25까지만 사용하기 때문).
- 결과 해석은 `reports/midterm/eda_findings.md`에 정리했다.
